# Phase 7 Validation: Optimization Modeling

This notebook runs a compact Phase 7 optimization sweep and previews outputs:
1. Efficiency + fairness-penalty model results
2. Robust max-min model results
3. Selected candidate bundles under a sample budget

In [ ]:
from pathlib import Path
import subprocess
import sys
import pandas as pd

repo_root = Path.cwd()
if not (repo_root / "src").exists():
    repo_root = Path.cwd().parent

solver_script = repo_root / "src" / "optimization" / "solver.py"
eff_path = repo_root / "data" / "processed" / "optimization_results_efficiency.csv"
robust_path = repo_root / "data" / "processed" / "optimization_results_robust.csv"

print("Repo root:", repo_root)
print("Solver script exists:", solver_script.exists())

In [ ]:
cmd = [
    sys.executable,
    str(solver_script),
    "--probability-profile",
    "Annual",
    "--budgets",
    "2000",
    "--lambdas",
    "0,1,3",
    "--max-seconds",
    "90",
]

run = subprocess.run(
    cmd,
    cwd=repo_root,
    capture_output=True,
    text=True,
    check=True,
)

print(run.stdout)
if run.stderr.strip():
    print("stderr:")
    print(run.stderr)

In [ ]:
eff_df = pd.read_csv(eff_path)
robust_df = pd.read_csv(robust_path)

print("Efficiency rows:", len(eff_df))
print("Robust rows:", len(robust_df))

eff_df

In [ ]:
robust_df

In [ ]:
preview = eff_df[["budget", "lambda", "objective_value", "gap", "selected_candidates_list"]].copy()
preview["selected_count_from_list"] = preview["selected_candidates_list"].fillna("").apply(
    lambda text: 0 if not text else len([token for token in str(text).split("|") if token])
)
preview